In [21]:
!pip install datasets transformers torch

In [22]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import IterableDataset, DataLoader
import torch

#### Lab 2 - Streaming LLM Data Pipeline

**Dataset:** `openwebtext` (streamed — never fully loaded into RAM)  
**Tokenizer:** `roberta-base`  

### Key Differences from Lab 1
| | Lab 1 | Lab 2 |
|---|---|---|
| Data loading | Full dataset in memory | Streaming, on-the-fly |
| Tokenization | Batched upfront | Lazy per example |
| Chunking | One-shot `group_texts` | Rolling buffer generator |
| Leftover tokens | Discarded | Padded and yielded |
| Scale | Small datasets | Web-scale corpora |

### Pipeline Overview
1. Load OpenWebText in streaming mode
2. Initialize RoBERTa tokenizer
3. Tokenize lazily on the fly
4. Rolling buffer generator → fixed-length 128-token blocks
5. Wrap in `IterableDataset` + `DataLoader`
6. Sanity-check batch shapes

In [23]:
"""
Streaming Language Modeling Data Pipeline — OpenWebText + RoBERTa
------------------------------------------------------------------
Goal:
    Build a *true streaming* LM pipeline that:
    - Processes OpenWebText without loading it into RAM.
    - Tokenizes with roberta-base on the fly.
    - Concatenates text and chunks into 128-token blocks via a rolling buffer.
    - Pads leftover tokens at the end of the stream.
    - Produces batches ready for PyTorch training.

Key Teaching Points:
    1. Streaming allows working with web-scale corpora (OpenWebText ~40GB).
    2. Rolling buffer handles document boundaries cleanly.
    3. Leftover padding ensures no tokens are wasted at stream end.
"""
print(__doc__)


Streaming Language Modeling Data Pipeline — OpenWebText + RoBERTa
------------------------------------------------------------------
Goal:
    Build a *true streaming* LM pipeline that:
    - Processes OpenWebText without loading it into RAM.
    - Tokenizes with roberta-base on the fly.
    - Concatenates text and chunks into 128-token blocks via a rolling buffer.
    - Pads leftover tokens at the end of the stream.
    - Produces batches ready for PyTorch training.

Key Teaching Points:
    1. Streaming allows working with web-scale corpora (OpenWebText ~40GB).
    2. Rolling buffer handles document boundaries cleanly.
    3. Leftover padding ensures no tokens are wasted at stream end.



In [24]:
# ============================================================
# 1. Load OpenWebText in STREAMING mode
# ============================================================
# streaming=True returns an IterableDataset.
# Data is fetched shard-by-shard as you iterate — zero RAM overhead.

stream_dataset = load_dataset(
    "openwebtext",
    split="train",
    streaming=True,
    trust_remote_code=True
)

# Peek at a single example without loading the rest
sample = next(iter(stream_dataset))
print(f"Sample text (first 200 chars):\n{sample['text'][:200]}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'openwebtext' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Sample text (first 200 chars):
Port-au-Prince, Haiti (CNN) -- Earthquake victims, writhing in pain and grasping at life, watched doctors and nurses walk away from a field hospital Friday night after a Belgian medical team evacuated


In [25]:
# ============================================================
# 2. Initialize the RoBERTa tokenizer
# ============================================================
# RoBERTa has a native <pad> token, so no workaround needed
# (unlike GPT-2 in the original labs).

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

print(f"Vocab size    : {tokenizer.vocab_size}")
print(f"Pad token     : '{tokenizer.pad_token}'  (id={tokenizer.pad_token_id})")
print(f"EOS token     : '{tokenizer.eos_token}'  (id={tokenizer.eos_token_id})")

Vocab size    : 50265
Pad token     : '<pad>'  (id=1)
EOS token     : '</s>'  (id=2)


In [26]:
# ============================================================
# 3. Lazy tokenization via streaming .map()
# ============================================================
# Unlike Lab 1 where .map() ran upfront on the full dataset,
# here it runs lazily as examples are consumed from the stream.

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        add_special_tokens=False   # Clean token stream for LM chunking
    )

tokenized_stream = stream_dataset.map(tokenize_function, batched=True)

print("Tokenized stream created (lazy — no data loaded yet).")

Tokenized stream created (lazy — no data loaded yet).


In [27]:
# ============================================================
# 4. Rolling buffer generator for fixed-length 128-token blocks
# ============================================================
# Key idea: maintain a buffer across document boundaries.
# Once the buffer has >= block_size tokens, yield a chunk.
# Leftover tokens at stream end are padded and yielded.

block_size = 128

def group_texts_streaming(dataset_iter, block_size, pad_token_id):
    buffer = []

    for example in dataset_iter:
        buffer.extend(example["input_ids"])

        while len(buffer) >= block_size:
            chunk  = buffer[:block_size]
            buffer = buffer[block_size:]
            yield {
                "input_ids"      : chunk,
                "attention_mask" : [1] * block_size
            }

    # Pad and yield leftover tokens (improvement over Lab 1)
    if buffer:
        pad_len = block_size - len(buffer)
        yield {
            "input_ids"      : buffer + [pad_token_id] * pad_len,
            "attention_mask" : [1] * len(buffer) + [0] * pad_len
        }

print(f"Rolling buffer generator defined. Block size = {block_size} tokens.")

Rolling buffer generator defined. Block size = 128 tokens.


In [28]:
# ============================================================
# 5. Wrap generator in a PyTorch IterableDataset
# ============================================================
# IterableDataset lets us plug the generator directly into
# a PyTorch DataLoader without materialising all data in memory.

class StreamingLMDataset(IterableDataset):
    def __init__(self, hf_iterable_dataset, block_size, pad_token_id):
        self.dataset     = hf_iterable_dataset
        self.block_size  = block_size
        self.pad_token_id = pad_token_id

    def __iter__(self):
        return group_texts_streaming(self.dataset, self.block_size, self.pad_token_id)


grouped_dataset = StreamingLMDataset(
    tokenized_stream,
    block_size=block_size,
    pad_token_id=tokenizer.pad_token_id
)

print("StreamingLMDataset created.")

StreamingLMDataset created.


In [29]:
# ============================================================
# 6. Collate function and DataLoader
# ============================================================
# All sequences are fixed-length (block_size), so collation is
# just a tensor stack — no dynamic padding needed.

def collate_fn(batch):
    input_ids      = torch.tensor([ex["input_ids"]      for ex in batch], dtype=torch.long)
    attention_mask = torch.tensor([ex["attention_mask"] for ex in batch], dtype=torch.long)
    return {
        "input_ids"      : input_ids,
        "attention_mask" : attention_mask,
        "labels"         : input_ids.clone()   # causal LM: predict next token
    }

train_loader = DataLoader(
    grouped_dataset,
    batch_size=8,
    collate_fn=collate_fn
    # Note: shuffle=True is NOT supported for IterableDataset
)

print("DataLoader ready.")

DataLoader ready.


In [30]:
# ============================================================
# 7. Sanity check — iterate over a few batches
# ============================================================

print("Sample streaming batches:\n")
for i, batch in enumerate(train_loader):
    print(f"Batch {i} → input_ids: {tuple(batch['input_ids'].shape)} | "
          f"attention_mask: {tuple(batch['attention_mask'].shape)} | "
          f"labels: {tuple(batch['labels'].shape)}")
    if i == 2:
        break

print()
print("Decoded first sequence of last batch (first 100 chars):")
print(tokenizer.decode(batch["input_ids"][0])[:100])

Sample streaming batches:



Token indices sequence length is longer than the specified maximum sequence length for this model (1217 > 512). Running this sequence through the model will result in indexing errors


Batch 0 → input_ids: (8, 128) | attention_mask: (8, 128) | labels: (8, 128)
Batch 1 → input_ids: (8, 128) | attention_mask: (8, 128) | labels: (8, 128)
Batch 2 → input_ids: (8, 128) | attention_mask: (8, 128) | labels: (8, 128)

Decoded first sequence of last batch (first 100 chars):
 Trump’s policy positions sounded like a preview of arguments to come.

“When we hear a candidate fo
